# Comparar herramientas actuales para RAG

Este cuaderno compara herramientas por la parte del pipeline que resuelven, no por marketing. La selección parte de documentación oficial consultada el **16 de julio de 2026**. La elección final se debe validar con el corpus, preguntas y restricciones reales del proyecto.

## 1. Mapa de decisión

| Capa | Alternativas comparadas | Pregunta correcta |
|---|---|---|
| Pipeline/chunking | Código propio, LangChain, LlamaIndex | ¿Necesito aprender/controlar cada paso o integrar conectores? |
| Vector DB | Chroma, Qdrant, pgvector, Weaviate, Pinecone | ¿Es un laboratorio local, una base Postgres existente o un servicio administrado/escalable? |
| Retrieval | denso, híbrido, filtros, re-ranking | ¿Las preguntas dependen de significado, palabras exactas, metadatos o una mezcla? |

**Principio:** no se cambia todo el stack a la vez. Primero se mide chunking y retrieval con preguntas fijas; después se justifica una dependencia nueva.

## 2. Herramientas de pipeline y segmentación

| Opción | Qué aporta | Cuándo usarla en este curso | Coste o límite |
|---|---|---|---|
| Código propio (este repo) | Transparencia: vemos texto, límites, solape, embeddings y coseno. | Primera implementación y laboratorio de chunking. | Hay que mantener loaders, metadatos y conectores. |
| [LangChain Text Splitters](https://docs.langchain.com/oss/python/integrations/splitters/index) | Splitters recursivos, por tokens y estructura. Su `RecursiveCharacterTextSplitter` baja de párrafos a oraciones y palabras. | Probar un baseline de chunking estándar sin escribirlo desde cero. | Abstrae detalles que aquí queremos enseñar; no mejora los documentos por sí solo. |
| [LlamaIndex](https://docs.llamaindex.ai/en/stable/module_guides/loading/ingestion_pipeline/) | Ingestion pipeline y transformaciones de documentos/nodos. | Proyectos que necesitan muchos conectores, parsing y composición de índices. | Añade un framework completo; no conviene para el primer RAG manual. |

Para los manuales de lavadora, el orden didáctico recomendado es: código propio → splitter recursivo de LangChain como control → estructura por encabezados cuando la extracción preserva los títulos.

## 3. Bases vectoriales y retrieval

| Herramienta | Fortaleza comprobable | Mejor encaje | Decisión para este laboratorio |
|---|---|---|---|
| [Chroma](https://docs.trychroma.com/docs/querying-collections/query-and-get) | Colecciones locales, persistencia y query con embeddings o textos; permite `where` sobre metadatos. | Notebook y demo local de un corpus pequeño/mediano. | **Elegida:** deja inspeccionar documentos, metadatos, vectores y distancias. |
| [Qdrant](https://qdrant.tech/documentation/search/hybrid-queries/) | Consultas híbridas/multietapa y fusión de dense + sparse. | Cuando se necesite combinar significado con términos/modelos/códigos exactos. | Siguiente experimento si las preguntas fallan por palabras clave. |
| [pgvector](https://github.com/pgvector/pgvector) | Búsqueda vectorial dentro de PostgreSQL, con joins, transacciones e índices HNSW/IVFFlat. | Datos y permisos ya viven en Postgres. | Elegirlo antes que otra base si el sistema empresarial ya usa Postgres. |
| [Weaviate](https://docs.weaviate.io/weaviate/search/hybrid) | Híbrido: vector + BM25F y fusión de resultados. | Aplicaciones con búsqueda híbrida administrada/autohospedada. | Candidato cuando se requiera híbrido como capacidad central. |
| [Pinecone](https://docs.pinecone.io/guides/search/hybrid-search) | Índices gestionados que almacenan dense + sparse por registro. | Producción gestionada con necesidad de operar poco la infraestructura. | No es necesario para el laboratorio local; evaluar coste, residencia y operación. |

No existe “la mejor” base universal. Todas deben evaluarse con el mismo conjunto de preguntas, filtros y métrica de retrieval.

## 4. Experimento mínimo y comparable

1. Mantén fijos los PDFs, el extractor y el modelo de embeddings.
2. Cambia sólo una variable: técnica de chunking **o** motor de retrieval.
3. Usa 10–15 preguntas con página/fuente esperada y 3 preguntas sin respuesta.
4. Registra Recall@k, MRR, tiempo de indexación, latencia de consulta y si la fuente recuperada permite responder.
5. Inspecciona fallos: un score alto no prueba que el fragmento tenga evidencia suficiente.

El laboratorio Streamlit complementa este cuaderno: permite observar primero los chunks, ajustar la estrategia y después ver qué retorna el RAG.

In [1]:
import pandas as pd

experimentos = pd.DataFrame([
    {'id': 'C01', 'variable': 'chunking', 'baseline': 'ventana por palabras', 'cambio': 'recursivo', 'métrica': 'Recall@4, MRR'},
    {'id': 'C02', 'variable': 'chunking', 'baseline': 'recursivo', 'cambio': 'por encabezados', 'métrica': 'Recall@4, citas válidas'},
    {'id': 'C03', 'variable': 'retrieval', 'baseline': 'Chroma dense', 'cambio': 'híbrido dense+sparse', 'métrica': 'Recall@4, latencia'},
    {'id': 'C04', 'variable': 'store', 'baseline': 'Chroma local', 'cambio': 'pgvector o Qdrant', 'métrica': 'latencia, filtros, operación'},
])
experimentos


,id,variable,baseline,cambio,métrica
0,C01,chunking,ventana por palabras,recursivo,"Recall@4, MRR"
1,C02,chunking,recursivo,por encabezados,"Recall@4, citas válidas"
2,C03,retrieval,Chroma dense,híbrido dense+sparse,"Recall@4, latencia"
3,C04,store,Chroma local,pgvector o Qdrant,"latencia, filtros, operación"


## Fuentes oficiales

- [LangChain: Text splitters](https://docs.langchain.com/oss/python/integrations/splitters/index)
- [LlamaIndex: Ingestion pipeline](https://docs.llamaindex.ai/en/stable/module_guides/loading/ingestion_pipeline/)
- [Chroma: Query and get](https://docs.trychroma.com/docs/querying-collections/query-and-get)
- [Qdrant: Hybrid and multi-stage queries](https://qdrant.tech/documentation/search/hybrid-queries/)
- [pgvector: documentación oficial](https://github.com/pgvector/pgvector)
- [Weaviate: Hybrid search](https://docs.weaviate.io/weaviate/search/hybrid)
- [Pinecone: Hybrid search](https://docs.pinecone.io/guides/search/hybrid-search)
